# ODE Report Generator

Select a built-in equation or edit the text, then click **Analyze Equation** to run the SINDy pipeline and generate a structured report.

**Run this notebook from the project root** (the `SINDy` folder) so that `simulations` and `sindy` can be imported.

In [1]:
import sys
import os
# Run notebook from project root (SINDy/) so simulations and sindy are importable
if os.path.basename(os.getcwd()) == "experiments":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, Latex, clear_output

# Simulations and SINDy pipeline (existing backend)
from simulations.exponential_decay import simulate_exponential_decay, true_coefficients_polynomial as true_exp
from simulations.logistic_growth import simulate_logistic_growth, true_coefficients_polynomial as true_logistic
from simulations.damped_harmonic_oscillator import (
    simulate_damped_harmonic_oscillator,
    true_coefficients_polynomial as true_damped,
)
from simulations.lorenz import simulate_lorenz, true_coefficients_polynomial as true_lorenz
from simulations.lotka_volterra import simulate_lotka_volterra, true_coefficients_polynomial as true_lv
from simulations.nonlinear_pendulum import simulate_nonlinear_pendulum, true_coefficients_polynomial as true_pendulum
from sindy.differentiate import estimate_derivative
from sindy.library import polynomial_library
from sindy.sparse_regression import sparse_regression
from sindy.sindy_model import print_equations
from sindy.metrics import coefficient_error, trajectory_reconstruction_error
from sindy.add_noise import add_noise

In [2]:
# Built-in systems: (label, equation_string, system_key)
BUILTIN_SYSTEMS = [
    ("Exponential decay", "dx/dt = -k*x", "exponential_decay"),
    ("Logistic growth", "dx/dt = r*x*(1 - x/K)", "logistic_growth"),
    ("Damped oscillator", "x'' + 2*beta*x' + omega^2*x = 0", "damped_oscillator"),
    ("Lorenz system", "dx/dt = sigma*(y-x), dy/dt = rho*x - y - x*z, dz/dt = xy - beta*z", "lorenz"),
    ("Lotka–Volterra", "dx/dt = a*x - b*x*y, dy/dt = -c*y + d*x*y", "lotka_volterra"),
    ("Nonlinear pendulum", "theta'' + (g/L)*sin(theta) = 0", "nonlinear_pendulum"),
]

def text_to_system_key(text):
    """Map equation text to a built-in system key (keyword match)."""
    t = (text or "").strip().lower()
    if "exponential" in t or "-k*x" in t or "dx/dt = -" in t and "x²" not in t and "x^2" not in t:
        return "exponential_decay"
    if "logistic" in t or "(1 - x" in t or "r*x*(1" in t or "x/k" in t:
        return "logistic_growth"
    if "damped" in t or "x''" in t or "omega" in t and ("beta" in t or "zeta" in t):
        return "damped_oscillator"
    if "lorenz" in t or "sigma*(y-x)" in t or "rho*x" in t and "xz" in t:
        return "lorenz"
    if "lotka" in t or "volterra" in t or "prey" in t or "a*x - b*x*y" in t:
        return "lotka_volterra"
    if "pendulum" in t or "sin(theta)" in t or "theta''" in t:
        return "nonlinear_pendulum"
    return None

In [3]:
def run_analysis(system_key, noise_std=0.0, random_seed=42):
    """Run SINDy pipeline for the given system. noise_std: add Gaussian noise to state data. Returns (report_md, figures)."""
    figures = []
    try:
        if system_key == "exponential_decay":
            k = 0.5
            t, y = simulate_exponential_decay(k=k, x0=1.0)
            if noise_std > 0:
                y = add_noise(y, noise_std=noise_std, random_seed=random_seed)
            dydt = estimate_derivative(t, y, method="savgol", window_length=11, polyorder=3)
            Theta, names = polynomial_library(y, degree=2)
            Xi = sparse_regression(Theta, dydt, threshold=0.05)
            Xi_true = true_exp(k, degree=2)
            state_names = ["x"]
            params = {"k": k}
            system_type = "Exponential decay"
            eq_latex = r"\frac{dx}{dt} = -kx"
            # trajectory
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(t, y[:, 0], "b-", label="x(t)")
            ax.set_xlabel("t"); ax.set_ylabel("x"); ax.set_title("Time series")
            ax.legend(); ax.grid(True, alpha=0.3)
            figures.append(fig)
        elif system_key == "logistic_growth":
            r, K = 2.0, 5.0
            t, y = simulate_logistic_growth(r=r, K=K, x0=0.5)
            if noise_std > 0:
                y = add_noise(y, noise_std=noise_std, random_seed=random_seed)
            dydt = estimate_derivative(t, y, method="savgol", window_length=11, polyorder=3)
            Theta, names = polynomial_library(y, degree=3)
            Xi = sparse_regression(Theta, dydt, threshold=0.05)
            Xi_true = true_logistic(r, K, degree=3)
            state_names = ["x"]
            params = {"r": r, "K": K}
            system_type = "Logistic growth"
            eq_latex = r"\frac{dx}{dt} = rx\left(1 - \frac{x}{K}\right) = rx - \frac{r}{K}x^2"
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(t, y[:, 0], "b-", label="x(t)")
            ax.set_xlabel("t"); ax.set_ylabel("x"); ax.set_title("Time series")
            ax.legend(); ax.grid(True, alpha=0.3)
            figures.append(fig)
            fig2, ax2 = plt.subplots(figsize=(6, 3))
            x_pos = np.arange(len(names))
            w = 0.35
            ax2.bar(x_pos - w/2, Xi_true[:, 0], w, label="True", color="C0")
            ax2.bar(x_pos + w/2, Xi[:, 0], w, label="Recovered", color="C1")
            ax2.set_xticks(x_pos); ax2.set_xticklabels(names)
            ax2.set_ylabel("Coefficient"); ax2.set_title("True vs recovered coefficients")
            ax2.legend(); ax2.grid(True, alpha=0.3, axis="y")
            figures.append(fig2)
        elif system_key == "damped_oscillator":
            beta, omega = 0.2, 2.0
            t, y = simulate_damped_harmonic_oscillator(beta=beta, omega=omega)
            if noise_std > 0:
                y = add_noise(y, noise_std=noise_std, random_seed=random_seed)
            dydt = estimate_derivative(t, y, method="savgol", window_length=11, polyorder=3)
            Theta, names = polynomial_library(y, degree=2)
            Xi = sparse_regression(Theta, dydt, threshold=0.05)
            Xi_true = true_damped(beta, omega, degree=2)
            state_names = ["x", "v"]
            params = {"beta": beta, "omega": omega, "omega_n": omega}
            system_type = "Damped harmonic oscillator"
            eq_latex = r"\ddot{x} + 2\beta\dot{x} + \omega^2 x = 0"
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(t, y[:, 0], label="x"); ax.plot(t, y[:, 1], label="v")
            ax.set_xlabel("t"); ax.set_ylabel("state"); ax.set_title("Time series")
            ax.legend(); ax.grid(True, alpha=0.3)
            figures.append(fig)
            fig2, ax2 = plt.subplots(figsize=(5, 4))
            ax2.plot(y[:, 0], y[:, 1]); ax2.set_xlabel("x"); ax2.set_ylabel("v")
            ax2.set_title("Phase portrait"); ax2.grid(True, alpha=0.3)
            figures.append(fig2)
        elif system_key == "lorenz":
            sigma, rho, beta = 10.0, 28.0, 8.0/3.0
            t, y = simulate_lorenz(sigma=sigma, rho=rho, beta=beta, num_points=2500)
            if noise_std > 0:
                y = add_noise(y, noise_std=noise_std, random_seed=random_seed)
            dydt = estimate_derivative(t, y, method="savgol", window_length=21, polyorder=3)
            Theta, names = polynomial_library(y, degree=2)
            Xi = sparse_regression(Theta, dydt, threshold=0.15)
            Xi_true = true_lorenz(sigma, rho, beta, degree=2)
            state_names = ["x", "y", "z"]
            params = {"sigma": sigma, "rho": rho, "beta": beta}
            system_type = "Lorenz system"
            eq_latex = r"\dot{x}=\sigma(y-x),\quad \dot{y}=\rho x - y - xz,\quad \dot{z}=xy-\beta z"
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.plot(t, y[:, 0], label="x", alpha=0.8)
            ax.plot(t, y[:, 1], label="y", alpha=0.8)
            ax.plot(t, y[:, 2], label="z", alpha=0.8)
            ax.set_xlabel("t"); ax.set_ylabel("state"); ax.set_title("Time series")
            ax.legend(); ax.grid(True, alpha=0.3)
            figures.append(fig)
        elif system_key == "lotka_volterra":
            a, b, c, d = 1.0, 0.5, 1.0, 0.5
            t, y = simulate_lotka_volterra(a=a, b=b, c=c, d=d)
            if noise_std > 0:
                y = add_noise(y, noise_std=noise_std, random_seed=random_seed)
            dydt = estimate_derivative(t, y, method="savgol", window_length=11, polyorder=3)
            Theta, names = polynomial_library(y, degree=2)
            Xi = sparse_regression(Theta, dydt, threshold=0.05)
            Xi_true = true_lv(a, b, c, d, degree=2)
            state_names = ["x (prey)", "y (predator)"]
            params = {"a": a, "b": b, "c": c, "d": d}
            system_type = "Lotka–Volterra"
            eq_latex = r"\dot{x}=ax-bxy,\qquad \dot{y}=-cy+dxy"
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(t, y[:, 0], label="prey"); ax.plot(t, y[:, 1], label="predator")
            ax.set_xlabel("t"); ax.set_ylabel("population"); ax.set_title("Time series")
            ax.legend(); ax.grid(True, alpha=0.3)
            figures.append(fig)
            fig2, ax2 = plt.subplots(figsize=(5, 4))
            ax2.plot(y[:, 0], y[:, 1]); ax2.set_xlabel("prey"); ax2.set_ylabel("predator")
            ax2.set_title("Phase plane"); ax2.grid(True, alpha=0.3)
            figures.append(fig2)
        elif system_key == "nonlinear_pendulum":
            g_over_L = 9.81
            t, y = simulate_nonlinear_pendulum(g_over_L=g_over_L, theta0=0.2)
            if noise_std > 0:
                y = add_noise(y, noise_std=noise_std, random_seed=random_seed)
            dydt = estimate_derivative(t, y, method="savgol", window_length=11, polyorder=3)
            Theta, names = polynomial_library(y, degree=3)
            Xi = sparse_regression(Theta, dydt, threshold=0.05)
            Xi_true = true_pendulum(g_over_L, degree=3)
            state_names = ["theta", "theta_dot"]
            params = {"g/L": g_over_L}
            system_type = "Nonlinear pendulum"
            eq_latex = r"\ddot{\theta} + \frac{g}{L}\sin\theta = 0"
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.plot(t, y[:, 0], label="theta")
            ax.plot(t, y[:, 1], label="theta_dot")
            ax.set_xlabel("t"); ax.set_ylabel("state"); ax.set_title("Time series")
            ax.legend(); ax.grid(True, alpha=0.3)
            figures.append(fig)
        else:
            return ([("markdown", "*Unsupported or unrecognized equation format. Please try one of the built-in examples or use a supported form.*")], [])

        coef_err = coefficient_error(Xi, Xi_true)
        try:
            recon_err = trajectory_reconstruction_error(Theta, Xi, dydt)
        except Exception:
            recon_err = None

        # Build recovered equation string (plain text for Markdown)
        recovered_lines = []
        for i in range(Xi.shape[1]):
            terms = []
            for j in range(Xi.shape[0]):
                if abs(Xi[j, i]) > 1e-8:
                    terms.append(f"{Xi[j, i]:.3f}*{names[j]}")
            recovered_lines.append(" + ".join(terms) if terms else "0")

        params_str = ", ".join(f"{k} = {v}" for k, v in params.items())
        if noise_std > 0:
            params_str += f", noise_std = {noise_std}"
        report_parts = [
            ("markdown", "## Equation Summary\n\nCanonical form:\n\n"),
            ("latex", eq_latex),
            ("markdown", f"""

---

## Detected System Type

{system_type}

---

## Extracted Parameters

{params_str}

---

## Recovered Governing Equation (SINDy)

"""
            ),
        ]
        for i, line in enumerate(recovered_lines):
            line_escaped = line.replace("*", "\\*")
            report_parts.append(("markdown", f"- **State {i+1}:** `{line_escaped}`  \n"))
        report_parts.append(("markdown", f"""

**Coefficient error (relative):** {coef_err:.6e}  
"""
        ))
        if recon_err is not None:
            report_parts.append(("markdown", f"**Trajectory reconstruction error:** {recon_err:.6e}\n"))
        report_parts.append(("markdown", """

---

## Interpretation

The SINDy pipeline (simulate → derivative estimation → polynomial library → STLSQ) recovered the governing equations from the time series. Small coefficient drift is expected from numerical derivative estimation and regression. For chaotic systems (e.g. Lorenz), a higher sparsity threshold reduces spurious terms.
"""
        ))
        return report_parts, figures
    except Exception as e:
        return ([("markdown", f"*Error during analysis: {str(e)}*  \n\nUnsupported or unrecognized equation format. Please try one of the built-in examples or use a supported form.")], [])

In [4]:
# UI layout
example_dropdown = widgets.Dropdown(
    options=[(label, key) for label, _, key in BUILTIN_SYSTEMS],
    value="logistic_growth",
    description="Example:",
    style={"description_width": "80px"},
)
equation_text = widgets.Textarea(
    value=next(eq for _, eq, k in BUILTIN_SYSTEMS if k == "logistic_growth"),
    placeholder="Enter equation (e.g. dx/dt = r*x*(1 - x/K))",
    description="Equation:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="100%", height="60px"),
)
noise_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=0.1,
    step=0.01,
    description="Noise (std):",
    style={"description_width": "80px"},
    readout_format=".2f",
)

def on_dropdown_change(change):
    eq = next(eq for _, eq, k in BUILTIN_SYSTEMS if k == change["new"])
    equation_text.value = eq

example_dropdown.observe(on_dropdown_change, names="value")

analyze_btn = widgets.Button(description="Analyze Equation", button_style="primary")
output_area = widgets.Output()

def on_analyze(btn):
    with output_area:
        clear_output(wait=True)
        text = equation_text.value.strip()
        # Prefer dropdown if it matches current text; else infer from text
        key = None
        for _, eq, k in BUILTIN_SYSTEMS:
            if k == example_dropdown.value and (not text or eq.strip() == text or text_to_system_key(text) == k):
                key = k
                break
        if key is None:
            key = example_dropdown.value or text_to_system_key(text)
        if not key:
            display(Markdown("*Unsupported or unrecognized equation format. Please try one of the built-in examples or use a supported form.*"))
            return
        report_parts, figs = run_analysis(key, noise_std=noise_slider.value)
        for kind, content in report_parts:
            if kind == "latex":
                display(Latex(f"$${content}$$"))
            else:
                display(Markdown(content))
        for fig in figs:
            plt.figure(fig.number)
            plt.show()

analyze_btn.on_click(on_analyze)

display(widgets.VBox([
    widgets.HTML("<b>Example</b>"),
    example_dropdown,
    widgets.HTML("<b>Equation (editable)</b>"),
    equation_text,
    widgets.HTML("<b>Noise</b> (Gaussian std added to state data; 0 = clean)"),
    noise_slider,
    analyze_btn,
    widgets.HTML("<hr><b>Report</b>"),
    output_area,
]))